In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from magenta.models.score2perf import score2perf


Instructions for updating:
non-resource variables are not supported in the long term


/opt/homebrew/Caskroom/miniforge/base/envs/music-transformer/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/music-transformer/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:31: UserWarning: WARN: A Box observation space has an unconventional shape (neither an image, nor a 1D vector). We recommend flattening the observation to have only a 1D vector or use a custom policy to properly process the data. Actual observation shape: (3, 3)
  logger.warn(


In [2]:
class PianoPerformanceLanguageModelProblem(score2perf.Score2PerfProblem):
  @property
  def add_eos_symbol(self):
    return True
  
problem = PianoPerformanceLanguageModelProblem()

from tensor2tensor import problems
unconditional_encoders = problem.get_feature_encoders()

from tensor2tensor.utils import trainer_lib
hparams = trainer_lib.create_hparams(hparams_set='transformer_tpu')
trainer_lib.add_problem_hparams(hparams, problem)
hparams.num_hidden_layers = 16
hparams.sampling_method = 'random'

from tensor2tensor.utils import decoding
decode_hparams = decoding.decode_hparams()
decode_hparams.alpha = 0.0
decode_hparams.beam_size = 1
ckpt_path = 'unconditional_model_16.ckpt'
run_config = trainer_lib.create_run_config(hparams)

estimator = trainer_lib.create_estimator(
    'transformer', hparams, run_config,
    decode_hparams=decode_hparams)

# def input_generator():
#     global targets
#     global decode_length
#     while True:
#         print('generating')
#         yield {
#             'targets': np.array([targets], dtype=np.int32),
#             'decode_length': np.array(decode_length, dtype=np.int32)
#         }
# targets = []
# decode_length = 0
# input_fn = decoding.make_input_fn_from_generator(input_generator())
# unconditional_samples = estimator.predict(
#     input_fn, checkpoint_path=ckpt_path)

/opt/homebrew/Caskroom/miniforge/base/envs/music-transformer/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
Use tf.keras instead.


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Configuring DataParallelism to replicate the model.


INFO:tensorflow:Configuring DataParallelism to replicate the model.


...

INFO:tensorflow:ps_devices: ['gpu:0']


Instructions for updating:
Use tf.keras instead.


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Using config: {'_model_dir': '/var/folders/cv/96wzgj4x73l9szhbb0cbhsqw0000gn/T/tmp9l8oxnme', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': 1000, '_save_checkpoints_secs': None, '_session_config': gpu_options {
  per_process_gpu_memory_fraction: 0.95
}
allow_soft_placement: true
graph_options {
  optimizer_options {
    global_jit_level: OFF
  }
}
isolate_session_state: true
, '_keep_checkpoint_max': 20, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas':

INFO:tensorflow:Using config: {'_model_dir': '/var/folders/cv/96wzgj4x73l9szhbb0cbhsqw0000gn/T/tmp9l8oxnme', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': 1000, '_save_checkpoints_secs': None, '_session_config': gpu_options {
  per_process_gpu_memory_fraction: 0.95
}
allow_soft_placement: true
graph_options {
  optimizer_options {
    global_jit_level: OFF
  }
}
isolate_session_state: true
, '_keep_checkpoint_max': 20, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas':

In [26]:
problem.get_hparams().get('input_space_id'), problem.get_hparams().get('target_space_id')

(0, 0)

In [27]:
def serving_input_receiver_fn():
    targets = tf.compat.v1.placeholder(dtype=tf.int64, shape=[None], name='targets')
    decode_length = tf.compat.v1.placeholder(dtype=tf.int32, shape=[], name='decode_length')
    receiver_tensors = {'targets': targets, 'decode_length': decode_length}
    return tf.estimator.export.ServingInputReceiver(receiver_tensors, receiver_tensors)

export_dir = 'exported_model'
estimator.export_saved_model(export_dir, serving_input_receiver_fn, checkpoint_path=ckpt_path)

INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Calling model_fn.


INFO:tensorflow:Setting T2TModel mode to 'infer'


INFO:tensorflow:Setting T2TModel mode to 'infer'


INFO:tensorflow:Setting hparams.dropout to 0.0


...

INFO:tensorflow:No assets to write.


INFO:tensorflow:No assets to write.


INFO:tensorflow:SavedModel written to: exported_model/temp-1683856236/saved_model.pb


INFO:tensorflow:SavedModel written to: exported_model/temp-1683856236/saved_model.pb


b'exported_model/1683856236'

In [1]:
_ = next(unconditional_samples) # burn one

targets = []
decode_length = 1024
sample_ids = next(unconditional_samples)['outputs']

from tensor2tensor.data_generators import text_encoder


import note_seq
import shutil
import os

def decode(ids, encoder):
    ids = list(ids)
    if text_encoder.EOS_ID in ids:
        idx = ids.index(text_encoder.EOS_ID)
        print('found EOS idx at', idx, 'out of', len(ids))
        ids = ids[:idx]
    else:
        print('did not find EOS in stream', len(ids))
    return encoder.decode(ids)

words = open('words.txt').read().splitlines()
import random
def random_name():
    return '-'.join(random.sample(words, 3))

while True:
    try:
        targets = []
        decode_length = 8192
        sample_ids = next(unconditional_samples)['outputs']

        midi_filename = decode(
            sample_ids,
            encoder=unconditional_encoders['targets'])

        unconditional_ns = note_seq.midi_file_to_note_sequence(midi_filename)

        output_fn = 'output/' + random_name() + '.mid'
        shutil.copyfile(midi_filename, output_fn)
        print(output_fn)

    except KeyboardInterrupt:
        pass

Instructions for updating:
non-resource variables are not supported in the long term


/opt/homebrew/Caskroom/miniforge/base/envs/music-transformer/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/music-transformer/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:31: UserWarning: WARN: A Box observation space has an unconventional shape (neither an image, nor a 1D vector). We recommend flattening the observation to have only a 1D vector or use a custom policy to properly process the data. Actual observation shape: (3, 3)
  logger.warn(
